In [1]:
import numpy as np
import pandas as pd
import pyfixest as pf
from duckreg import duckreg as dr

In [2]:
adm2 = pd.read_parquet("../../data_nobackup/assembled/adm2_1km.parquet")
adm2

,GID_2,year,modis_median,modis_mean,modis_rollmax3,modis_gt30C,avhrr_median,avhrr_mean,avhrr_rollmax3,avhrr_gt30C,...,WB_LM_2011,HDI_HI_1991,HDI_VH_2011,HDI_ME_1999,HDI_ME_2011,WB_LO_1999,WB_LO_1991,WB_UM_1999,HDI_HI_2011,HDI_ME_1991
0,?,1992,NaN,NaN,NaN,NaN,297.510801,293.223161,299.250000,3.715258,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,?,1993,NaN,NaN,NaN,NaN,295.320791,291.433954,296.991397,1.420102,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,?,1994,NaN,NaN,NaN,NaN,299.428378,296.133851,302.698709,4.137015,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,?,1995,NaN,NaN,NaN,NaN,293.815556,293.842332,296.296003,5.293454,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,?,1996,NaN,NaN,NaN,NaN,298.687614,296.591203,299.016423,2.230486,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1407445,ZWE.9.9_2,2017,293.313162,293.248823,301.949119,1.176511,299.017619,299.697024,310.401510,42.780825,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1407446,ZWE.9.9_2,2018,294.139738,294.447412,303.326284,3.279089,298.539072,298.454745,310.745245,26.846563,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1407447,ZWE.9.9_2,2019,296.038759,295.562615,303.419225,8.968683,299.357705,298.967782,300.540393,3.165924,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1407448,ZWE.9.9_2,2020,294.493232,294.114313,303.707066,4.173461,298.792838,299.208201,NaN,1.000000,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [4]:
adm2[["GID_2", "subdivision"]].nunique()

GID_2          46915
subdivision     2846
dtype: int64

In [4]:
adm2 = (
    adm2
    .assign(
        log_ntl_harm=lambda x: np.log(x["ntl_harm"] + 0.01),
        leader=lambda x: x["adm2_reg_fav"],
        lag_leader=lambda x: (
            x.groupby("GID_2")["adm2_reg_fav"].shift(1)
        ),
    )
)

In [ ]:
fit = pf.feols(
    "log_ntl_harm ~ leader | GID_2 + country^year",
    data = adm2
)

fit.summary()

In [6]:
fit = pf.feols(
    "log_ntl_harm ~ lag_leader | GID_2 + country^year",
    data = adm2
)

fit.summary()

/scicore/home/meiera/schulz0022/miniforge-pypy3/envs/gnt/lib/python3.11/site-packages/pyfixest/estimation/model_matrix_fixest_.py:215: UserWarning: 116 singleton fixed effect(s) detected. These observations are dropped from the model.
  warnings.warn(


###

Estimation:  OLS
Dep. var.: log_ntl_harm, Fixed effects: GID_2+country^year
Inference:  iid
Observations:  1359752

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| lag_leader    |      0.070 |        0.015 |     4.804 |      0.000 |  0.041 |   0.098 |
---
RMSE: 0.592 R2: 0.938 R2 Within: 0.0 


In [8]:
fit = dr(
    "log_ntl_harm ~ lag_leader | GID_2 + country^year",
    data = adm2
)

fit.print_summary()

RegressionResults(coefficients=array([0.06991498]), coef_names=['lag_leader'], vcov=array([[8.02263082e-05]]), n_obs=1359752, n_compressed=14327, se_type='HC1', duckreg_version='0.4.5', computed_at='2026-07-10T10:50:23.745615')


In [15]:
fit = dr(
    "log(ntl_harm + 0.01) ~ lag_leader | subdivision + country^year",
    data = adm2
)

fit.print_summary()

RegressionResults(coefficients=array([0.97716748]), coef_names=['lag_leader'], vcov=array([[0.00062705]]), n_obs=1359752, n_compressed=22992, se_type='HC1', duckreg_version='0.4.5', computed_at='2026-07-10T11:59:27.576029')
